# Comparação dos métodos de segmentação do pipeline refinado

Compara normal+RG, fuzzy+RG, normal+FC e fuzzy+FC com a mesma configuração
base: -300 HU, P99.9, filtro robusto, envelope da aorta e surface padding 2.
Os métodos de threshold e segmentação são identificados pelas colunas por exame.

Os runs atuais contêm treino (30), validação (60) e teste (700), analisados
separadamente. A comparação principal usa teste; a melhor alternativa nesse
conjunto é descritiva e não deve orientar ajuste de parâmetros.

O cenário conservador inclui falhas registradas com Dice zero. A média dos
casos avaliáveis é mostrada separadamente para reproduzir os resumos do sweep.


In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = next(
    root
    for root in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (root / "src" / "utils").is_dir()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.comparison_utils.paired_statistics import adjust_holm, compare_paired_dice
from utils.comparison_utils.variant_comparison import (
    build_delta_summary_vs_reference,
    build_dice_stats_by_variant,
)
from utils.project.results.schema import normalize_result_status
from utils.project.runtime.notebook_env import configure_notebook_environment

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
RESULT_ROOT = (
    REPO_ROOT / "output/segmentation/runs/mid_res/fuzzy_comparison/current_pipeline"
)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

### Funções de apresentação
Os helpers abaixo organizam os gráficos específicos desta análise.

In [ ]:
from eda.comparisons.view.segmentation_method_views import (
    plot_ostia_status_by_variant,
    plot_pair_delta_by_image,
    plot_pair_dice_by_image,
)

## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento dos Resultados

Os CSVs `results/image_results.csv` dos runs `train30_val60` e `test700`
contêm os quatro métodos. `ANALYSIS_SPLIT` permite selecionar treino, validação
ou teste sem misturar as coortes. O panorama por conjunto aparece abaixo.


In [ ]:
ANALYSIS_SPLIT = "test"  # "train", "val" ou "test"
PREFERRED_ORDER = [
    "normal_rg",
    "fuzzy_threshold_rg",
    "normal_threshold_fc",
    "fuzzy_threshold_fc",
]
PRETTY_NAMES = {
    "normal_rg": "Normal + RG (referência)",
    "fuzzy_threshold_rg": "Fuzzy threshold + RG",
    "normal_threshold_fc": "Normal + FC",
    "fuzzy_threshold_fc": "Fuzzy threshold + FC",
}
RUN_DIRS = {
    "train_val": RESULT_ROOT / "train30_val60",
    "test": RESULT_ROOT / "test700",
}

In [ ]:
# Carrega os resultados atuais sem usar os rótulos históricos do ranking.
all_results_df = pd.concat(
    [
        pd.read_csv(run_dir / "results/image_results.csv")
        for run_dir in RUN_DIRS.values()
    ],
    ignore_index=True,
)
if ANALYSIS_SPLIT not in {"train", "val", "test"}:
    raise ValueError("ANALYSIS_SPLIT deve ser train, val ou test.")
if all_results_df.duplicated(["split", "variant", "IMG_ID"]).any():
    raise ValueError("Há exames duplicados por conjunto e variante.")
if set(all_results_df["variant"]) != set(PREFERRED_ORDER):
    raise ValueError("As variantes salvas não correspondem aos quatro métodos.")

# Zero representa uma falha registrada; ausências sem diagnóstico são rejeitadas.
all_results_df["recorded_dice"] = pd.to_numeric(
    all_results_df["dice_artery"], errors="raise"
)
failure_mask = (
    all_results_df["recorded_dice"].isna()
    & all_results_df["error"].notna()
    & ~all_results_df["segmentation_attempted"]
)
if (all_results_df["recorded_dice"].isna() & ~failure_mask).any():
    raise ValueError("Há Dice ausente sem falha registrada do pipeline.")
all_results_df["artery_dice"] = all_results_df["recorded_dice"].fillna(0.0)
all_results_df["folder_variant"] = all_results_df["variant"]
all_results_df["variant_label"] = all_results_df["variant"].map(PRETTY_NAMES)
all_results_df["ostia_detection_status"] = all_results_df["ostia_status"].map(
    normalize_result_status
)
all_results_df["found_wrong"] = all_results_df["ostia_detection_status"].isin(
    ["one_correct", "none_correct", "found_but_wrong"]
)
all_results_df["not_found_or_error"] = (
    ~all_results_df["ostia_success"] & ~all_results_df["found_wrong"]
)

# Cada conjunto deve conter os mesmos IDs em todas as variantes.
expected_counts = {"train": 30, "val": 60, "test": 700}
for split, count in expected_counts.items():
    split_frame = all_results_df.loc[all_results_df["split"].eq(split)]
    ids_by_variant = [
        set(split_frame.loc[split_frame["variant"].eq(variant), "IMG_ID"])
        for variant in PREFERRED_ORDER
    ]
    if any(len(ids) != count or ids != ids_by_variant[0] for ids in ids_by_variant):
        raise ValueError(f"Coorte incompleta ou diferente entre métodos: {split}")

overview_df = all_results_df.groupby(["split", "folder_variant"], as_index=False).agg(
    n_images=("IMG_ID", "size"),
    valid_dice_n=("recorded_dice", "count"),
    mean_dice=("artery_dice", "mean"),
    mean_dice_evaluable=("recorded_dice", "mean"),
    ostia_success_n=("ostia_success", "sum"),
    ostia_success_rate=("ostia_success", "mean"),
)
results_df = all_results_df.loc[all_results_df["split"].eq(ANALYSIS_SPLIT)].copy()
summary_df = results_df.groupby(
    ["folder_variant", "variant_label"], as_index=False
).agg(
    n_images=("IMG_ID", "size"),
    valid_dice_n=("recorded_dice", "count"),
    mean_dice=("artery_dice", "mean"),
    mean_dice_evaluable=("recorded_dice", "mean"),
    median_dice=("artery_dice", "median"),
    std_dice=("artery_dice", "std"),
    ostia_success_rate=("ostia_success", "mean"),
    both_correct_n=("both_correct", "sum"),
    both_tolerable_n=("both_tolerable", "sum"),
    found_wrong_n=("found_wrong", "sum"),
    not_found_or_error_n=("not_found_or_error", "sum"),
)
print(f"Conjunto principal: {ANALYSIS_SPLIT}")
print(f"Exames por método: {expected_counts[ANALYSIS_SPLIT]}")

In [ ]:
# Treino, validação e teste permanecem separados no panorama geral.
display(overview_df.round(4))

## Detecção dos Óstios

Aqui o status é separado em: ambos corretos, ambos toleráveis, encontrados porém incorretos e não encontrado/erro.

In [ ]:
plot_ostia_status_by_variant(
    summary_df,
    preferred_order=PREFERRED_ORDER,
    pretty_names=PRETTY_NAMES,
    save_path=None,
)
plt.show()

## Dice Score por Variante

A tabela abaixo resume o Dice arterial por variante. O gráfico em seguida mostra média e mediana para comparação visual.

In [ ]:
dice_plot = summary_df.copy()
dice_plot["variant_label"] = pd.Categorical(
    dice_plot["variant_label"],
    [PRETTY_NAMES.get(name, name) for name in PREFERRED_ORDER],
    ordered=True,
)
dice_plot = dice_plot.sort_values("variant_label")

In [ ]:
dice_stats_df = build_dice_stats_by_variant(results_df, PREFERRED_ORDER)

display(
    dice_stats_df.round(
        {
            "mean_dice": 4,
            "max_dice": 4,
            "min_dice": 4,
            "std_dice": 4,
            "median_dice": 4,
        }
    )
)

In [ ]:
x = np.arange(len(dice_plot))
width = 0.38
fig, ax = plt.subplots(figsize=(12, 5))
mean_bars = ax.bar(
    x - width / 2, dice_plot["mean_dice"], width, label="Média", color="#4c78a8"
)
median_bars = ax.bar(
    x + width / 2, dice_plot["median_dice"], width, label="Mediana", color="#f58518"
)

for bars in [mean_bars, median_bars]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.008,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

ax.set_xticks(x)
ax.set_xticklabels(dice_plot["variant_label"].astype(str), rotation=35, ha="right")
ax.set_ylabel("Dice arterial", fontsize=12)
y_max = max(0.75, float(dice_plot[["mean_dice", "median_dice"]].max().max()) + 0.08)
ax.set_ylim(0, y_max)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Melhor variante versus baseline

A melhor variante é escolhida pelo maior Dice médio entre as alternativas. A
comparação pareada usa `Normal + RG` como baseline e somente IDs presentes nos
dois resultados. O delta é calculado como `melhor variante - baseline`.

In [ ]:
baseline_variant = "normal_rg"
alternative_summary = summary_df.loc[
    summary_df["folder_variant"].astype(str) != baseline_variant
].copy()
BEST_COMPARISON_VARIANT = str(
    alternative_summary.sort_values("mean_dice", ascending=False).iloc[0][
        "folder_variant"
    ]
)

best_vs_baseline_df = build_delta_summary_vs_reference(
    results_df,
    baseline_variant,
    variants=[BEST_COMPARISON_VARIANT],
    pretty_names=PRETTY_NAMES,
)

print(f"Baseline: {PRETTY_NAMES[baseline_variant]}")
print(
    "Melhor variante: "
    f"{PRETTY_NAMES.get(BEST_COMPARISON_VARIANT, BEST_COMPARISON_VARIANT)}"
)
display(
    best_vs_baseline_df.round(
        {
            "mean_delta": 4,
            "median_delta": 4,
            "max_gain": 4,
            "max_loss": 4,
        }
    )
)

In [ ]:
DICE_PANEL_NAMES = {
    baseline_variant: "Baseline",
    BEST_COMPARISON_VARIANT: "Melhor variante",
}
plot_pair_dice_by_image(
    results_df,
    baseline_variant,
    BEST_COMPARISON_VARIANT,
    pretty_names=DICE_PANEL_NAMES,
    save_path=None,
)
plt.show()

In [ ]:
plot_pair_delta_by_image(
    results_df,
    baseline_variant,
    BEST_COMPARISON_VARIANT,
    pretty_names=PRETTY_NAMES,
    save_path=None,
)
plt.show()

## Comparações estatísticas pareadas

Wilcoxon bilateral contra normal+RG, com Holm para as três alternativas no
conjunto selecionado. O efeito é rank-biserial: positivo favorece a alternativa.
O IC95% da diferença média usa bootstrap pareado (10.000 amostras, seed 42).
O teste avalia ranks e o IC avalia a média; suas conclusões podem diferir.


In [ ]:
# Mantém todos os IDs, incluindo as falhas registradas como Dice zero.
baseline_df = results_df.loc[results_df["folder_variant"].eq("normal_rg")]
statistic_rows = []
for variant in PREFERRED_ORDER[1:]:
    candidate_df = results_df.loc[results_df["folder_variant"].eq(variant)]
    statistic_rows.append(
        {"variant": variant, **compare_paired_dice(baseline_df, candidate_df)}
    )
paired_statistics_df = pd.DataFrame(statistic_rows)
paired_statistics_df["p_holm"] = adjust_holm(paired_statistics_df["p_value"])
display(
    paired_statistics_df[
        [
            "variant",
            "paired_images",
            "mean_delta_dice",
            "wilcoxon_statistic",
            "p_value",
            "p_holm",
            "rank_biserial_effect",
            "mean_delta_ci_95_low",
            "mean_delta_ci_95_high",
        ]
    ]
)